# 04a — Tuning Hyperparameter RM-a (Full Fine-tuning) — versi **local**
Eksplorasi adaptif hyperparameter RM-a. Setiap keputusan berbasis hasil trial sebelumnya.

### Metodologi: eksplorasi adaptif (coordinate line-search)

Bukan grid buta. Mulai dari **baseline**, lalu tune satu hyperparameter menurut **urutan dampak**; tiap langkah **probe arah** (naik/turun) dan lanjut selama **val F1-macro** membaik (early-stop `eps=0.001`). HP berikutnya memakai config yang sudah diperbarui. **Setiap konfigurasi tersimpan** sebagai satu trial + `rationale` (alasan). Seleksi di validation; **test dievaluasi sekali** untuk config final. Semua hasil → `results/tuning/`.

## 1. Setup

In [ ]:
import os, sys
os.environ.setdefault('HF_HUB_OFFLINE','1'); os.environ.setdefault('TRANSFORMERS_OFFLINE','1')
os.environ.setdefault('MPLBACKEND','Agg')
from pathlib import Path
PROJECT_DIR = Path('..')                      # dijalankan dari notebooks/ (lokal)
sys.path.insert(0, str((PROJECT_DIR/'src').resolve()))
import numpy as np, pandas as pd, torch
import tuning as T
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device, '| lokal')

## 2. Ruang pencarian & baseline (bisa diedit)

In [ ]:
OUT_DIR = PROJECT_DIR / 'results' / 'tuning'
DATA_DIR = PROJECT_DIR / 'dataset' / 'splits'
META = PROJECT_DIR / 'dataset' / 'processed' / 'metadata.json'
import json as _json
ORDER   = ['lr','batch','warmup_ratio','weight_decay']   # urutan prioritas dampak
SPACES  = {'lr':[1e-5,2e-5,3e-5,5e-5], 'batch':[8,16,32],
           'warmup_ratio':[0.0,0.1,0.2], 'weight_decay':[0.0,0.01,0.1]}
BASELINE= {'lr':2e-5,'batch':16,'warmup_ratio':0.1,'weight_decay':0.01,
           'epochs':5,'micro_batch':16,'seed':42}   # micro_batch=16 -> batch 32 via grad-accum (aman 4GB)
print('order', ORDER); print('spaces', SPACES); print('baseline', BASELINE)

## 3. Data + tokenizer + class weights

In [ ]:
import json
from dataset import load_tokenizer
train_df = pd.read_csv(DATA_DIR/'train.csv'); val_df = pd.read_csv(DATA_DIR/'val.csv'); test_df = pd.read_csv(DATA_DIR/'test.csv')
cw = json.load(open(META, encoding='utf-8'))['class_weights']
weight = torch.tensor([cw['0'], cw['1']], dtype=torch.float, device=device)
tokenizer = load_tokenizer('indobenchmark/indobert-base-p2')
ctx = {'train_df':train_df,'val_df':val_df,'test_df':test_df,'tokenizer':tokenizer,'weight':weight,
       'device':device,'model_name':'indobenchmark/indobert-base-p2','max_length':128,'num_workers':0}
print('data', len(train_df), len(val_df), len(test_df))

## 4. Jalankan tuning adaptif (mahal — tiap trial ~beberapa menit)

In [ ]:
final_cfg, decisions, test_metrics = T.tune_rma(ctx, out_dir=str(OUT_DIR),
                                                spaces=SPACES, order=ORDER, baseline=BASELINE, eps=0.001)

## 5. Hasil, jejak keputusan, metrik test

In [ ]:
trials = pd.read_csv(OUT_DIR / 'rma_trials.csv')
cols = [c for c in ['trial_id','stage','val_f1_macro','val_f1_judi','best_epoch','train_time_s','rationale'] if c in trials.columns]
print(f'Total trial: {len(trials)}')
print(trials[cols].to_string(index=False))
print('\n=== Jejak keputusan ===')
for d in decisions:
    print(f"  [{d['stage']}] -> {d['chosen']}  (val F1 {d['val_f1_macro']:.4f})  | {d['note']}")
print('\n=== Config final ===', final_cfg)
print('=== Metrik TEST final ===')
for k in ['accuracy','f1_macro','precision_macro','recall_macro','f1_class1','precision_class1','recall_class1']:
    print(f'  {k:18s}: {test_metrics[k]:.4f}')
print('\nArtefak tersimpan di', OUT_DIR)

## Ringkasan
Semua trial + kurva line-search + checkpoint terbaik + `tuning_summary.json` tersimpan di `results/tuning/`. Config final RM-a dipilih by val F1-macro dengan alasan tercatat per tahap.